<a href="https://colab.research.google.com/github/authorsunilsir/Projects/blob/main/local_food_wastage_management_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import sqlite3

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Load data from CSV files
providers = pd.read_csv("/content/drive/MyDrive/providers_data.csv")
receivers = pd.read_csv("/content/drive/MyDrive/receivers_data.csv")
food_listings = pd.read_csv("/content/drive/MyDrive/food_listings_data.csv")
claims = pd.read_csv("/content/drive/MyDrive/claims_data.csv")

In [ ]:
providers

,Provider_ID,Name,Type,Address,City,Contact
0,1,Gonzales-Cochran,Supermarket,"74347 Christopher Extensions\nAndreamouth, OK ...",New Jessica,+1-600-220-0480
1,2,"Nielsen, Johnson and Fuller",Grocery Store,"91228 Hanson Stream\nWelchtown, OR 27136",East Sheena,+1-925-283-8901x6297
2,3,Miller-Black,Supermarket,"561 Martinez Point Suite 507\nGuzmanchester, W...",Lake Jesusview,001-517-295-2206
3,4,"Clark, Prince and Williams",Grocery Store,"467 Bell Trail Suite 409\nPort Jesus, IA 61188",Mendezmouth,556.944.8935x401
4,5,Coleman-Farley,Grocery Store,"078 Matthew Creek Apt. 319\nSaraborough, MA 53978",Valentineside,193.714.6577
...,...,...,...,...,...,...
995,996,"Vasquez, Ruiz and Flowers",Restaurant,"84308 Justin Stravenue\nNew Amberside, NE 53447",Williamview,+1-319-378-7627x0682
996,997,Garza-Williams,Catering Service,"08864 Figueroa Radial Suite 948\nJennaberg, AZ...",East Rossside,001-924-441-3963x746
997,998,Novak Group,Grocery Store,"934 Zachary Run\nMelissamouth, WY 02729",Joshuastad,(903)642-1969x3300
998,999,Moody Ltd,Grocery Store,"17580 Ernest Hills\nLake Michaelmouth, OR 56416",Stevenchester,637.300.3664x4880


In [ ]:
import plotly.express as px

# Count the occurrences of each provider type
provider_type_counts = providers['Type'].value_counts().reset_index()
provider_type_counts.columns = ['Provider_Type', 'Count']

# Create an interactive bar chart using Plotly Express
fig = px.bar(provider_type_counts, x='Provider_Type', y='Count', title='Distribution of Provider Types', color='Provider_Type')
fig.show()

In [ ]:
# Database connection
conn = sqlite3.connect("food_wastage.db")
cursor = conn.cursor()

In [ ]:
# Database connection
conn = sqlite3.connect("food_wastage.db")
cursor = conn.cursor()

# Creating tables
cursor.execute('''
    CREATE TABLE IF NOT EXISTS providers (
        Provider_ID INTEGER PRIMARY KEY,
        Name TEXT,
        Type TEXT,
        Address TEXT,
        City TEXT,
        Contact TEXT
    )
''')

In [ ]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS providers (
        Provider_ID INT PRIMARY KEY,
        Name VARCHAR(255),
        Type VARCHAR(100),
        Address TEXT,
        City VARCHAR(100),
        Contact VARCHAR(50)
    )
""")
conn.commit()

In [ ]:
# Insert data using to_sql
providers.to_sql('providers', conn, if_exists='replace', index=False)

1000

📌 Which city has the highest number of food providers?

In [ ]:
# SQL query to get the city with the highest number of providers
query = """
    SELECT City, COUNT(*) AS Provider_Count
    FROM providers
    GROUP BY City
    ORDER BY Provider_Count DESC
    LIMIT 1;
"""

cursor.execute(query)
result = cursor.fetchall()

# Convert result into a DataFrame for better readability
df = pd.DataFrame(result, columns=["City", "Provider_Count"])
df



,City,Provider_Count
0,South Christopherborough,3


In [ ]:
import plotly.express as px

# Count providers per city
city_provider_counts = providers['City'].value_counts().reset_index()
city_provider_counts.columns = ['City', 'Provider_Count']

# Create an interactive scatter map
fig = px.scatter_geo(city_provider_counts,
                     locations="City",
                     locationmode="country names",  # Or 'usa-states', 'world' etc. depending on your data's scope
                     size="Provider_Count",
                     hover_name="City",
                     title="Distribution of Food Providers by City",
                     color="City") # Add color based on city
fig.show()

In [ ]:
# Creating tables for receivers, food_listings, and claims
cursor.execute('''
    CREATE TABLE IF NOT EXISTS receivers (
        Receiver_ID INTEGER PRIMARY KEY,
        Name TEXT,
        Type TEXT,
        City TEXT,
        Contact TEXT
    )
''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS food_listings (
        Food_ID INTEGER PRIMARY KEY,
        Food_Name TEXT,
        Quantity INTEGER,
        Expiry_Date TEXT,
        Provider_ID INTEGER,
        Provider_Type TEXT,
        Location TEXT,
        Food_Type TEXT,
        Meal_Type TEXT,
        FOREIGN KEY (Provider_ID) REFERENCES providers(Provider_ID)
    )
''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS claims (
        Claim_ID INTEGER PRIMARY KEY,
        Food_ID INTEGER,
        Receiver_ID INTEGER,
        Status TEXT,
        Timestamp TEXT,
        FOREIGN KEY (Food_ID) REFERENCES food_listings(Food_ID),
        FOREIGN KEY (Receiver_ID) REFERENCES receivers(Receiver_ID)
    )
''')

conn.commit()

📌 Here are some more SQL queries you can use to explore the data:

*   **Get the total number of food listings per provider type:**

In [ ]:
# SQL query to get the top 10 most claimed food listings
query = """
    SELECT
        fl.Food_Name,
        COUNT(c.Claim_ID) AS TotalClaims
    FROM
        food_listings fl
    JOIN
        claims c ON fl.Food_ID = c.Food_ID
    GROUP BY
        fl.Food_Name
    ORDER BY
        TotalClaims DESC
    LIMIT 10; -- Get the top 10 most claimed
"""

cursor.execute(query)
result = cursor.fetchall()

# Convert result into a DataFrame for better readability
df_top_claimed_foods = pd.DataFrame(result, columns=["Food_Name", "TotalClaims"])
display(df_top_claimed_foods)

,Food_Name,TotalClaims
0,Rice,122
1,Soup,114
2,Dairy,110
3,Fish,108
4,Salad,106
5,Chicken,102
6,Bread,94
7,Pasta,87
8,Vegetables,86
9,Fruits,71


In [ ]:
import plotly.express as px

# Create an interactive bar chart for top claimed foods
fig = px.bar(df_top_claimed_foods, x='Food_Name', y='TotalClaims', title='Top 10 Most Claimed Food Listings', color='Food_Name')
fig.show()

In [ ]:
# SQL query to get the number of claims by status
query = """
    SELECT
        Status,
        COUNT(*) AS NumberOfClaims
    FROM
        claims
    GROUP BY
        Status;
"""

cursor.execute(query)
result = cursor.fetchall()

# Convert result into a DataFrame for better readability
df_claims_status = pd.DataFrame(result, columns=["Status", "NumberOfClaims"])
display(df_claims_status)

,Status,NumberOfClaims
0,Cancelled,336
1,Completed,339
2,Pending,325


In [ ]:
import plotly.express as px

# Create an interactive bar chart for claims status
fig = px.bar(df_claims_status, x='Status', y='NumberOfClaims', title='Distribution of Claims by Status', color='Status')
fig.show()

In [ ]:
# Insert data into the tables
receivers.to_sql('receivers', conn, if_exists='replace', index=False)
food_listings.to_sql('food_listings', conn, if_exists='replace', index=False)
claims.to_sql('claims', conn, if_exists='replace', index=False)

1000

In [ ]:
%%writefile foods_app.py
import streamlit as st

st.title("This is my project")

st.header("Local Food Wastage Management System")
st.subheader("This is a subheader")


st.text("This is plain text")
st.markdown("### This is a markdown text")


# Input Widgets
name = st.text_input("Enter your name")
feedback = st.text_area("Provide your feedback")



# Buttons and Checkboxes
if st.button("Submit"):
    st.success(f"Hello {name}, thanks for your feedback!")


agree = st.checkbox("I agree to the terms and conditions")
if agree:
    st.write("You agreed!")

choice = st.radio("Choose an option:", ["Option 1", "Option 2", "Option 3"])
st.write(f"You selected: {choice}")

# Selectbox and Multiselect
color = st.selectbox("Select a color", ["Red", "Green", "Blue"])
st.write(f"You chose: {color}")

hobbies = st.multiselect("Select your hobbies", ["Reading", "Coding", "Gaming"])
st.write(f"Your hobbies: {', '.join(hobbies)}")

# Sliders and Number Input
age = st.slider("Select your age", 1, 100, 25)
st.write(f"Your age is: {age}")


Overwriting foods_app.py
